# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [16]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from nba_api.stats.static import players

from src.config import *
from src.utils import *
from src.feature_builder import *
from src.feature_aggregation import *

#display full columns
pd.set_option('display.max_column', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_seq_items', None)
# pd.set_option('display.max_colwidth', 500)
# pd.set_option('expand_frame_repr', True)

In [17]:
#store start time of notebook
start_time = datetime.now()
print("Start time: ", start_time)

Start time:  2025-06-13 00:11:46.520878


# 🔁 Chargement des fichiers

In [18]:
boxscores_file = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)
games_file = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)
df_boxscores = pd.read_csv(boxscores_file, dtype={'gameId': str})
df_games = pd.read_csv(games_file, dtype={'GAME_ID': str})


In [19]:
games_file

'data\\raw_last\\games_merged\\games_merged_all_seasons_2025-06-13_00-03-39.csv'

In [20]:
df_boxscores

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage
0,0029900011,1610612747,Los Angeles,Lakers,LAL,lakers,779,Glen,Rice,G. Rice,glen-rice,F,NaN,NaN,40:11,102.6,88.2,15.5,14.5,0.150,0.75,13.0,0.026,0.143,0.081,17.4,0.808,0.871,0.225,89.63,90.80,76.0,0.224,0.500,0.349,0.201,0.440,0.328,0.226,0.258,2,5,10,0,14,14,9,28,1,1,0.538,0.462,0.214,0.214,0.536,0.357,0.250,0.071,0.000,1.000,0.000,1.0,0.0,1.000,0.000,8,13,5,6,0.833,7,7,1.000,1,5,6,3,2,0,4,5,28,11.0,0.286,0.206,0.714,0.545,0.467,0.318,0.077,0.217,0.167,0.143,0.267,0.222,0.00,0.200,0.263,0.045,0.359
1,0029900011,1610612747,Los Angeles,Lakers,LAL,lakers,920,A.C.,Green,A. Green,ac-green,F,NaN,NaN,32:55,111.5,96.7,18.1,14.8,0.087,0.00,28.6,0.100,0.130,0.113,0.0,0.200,0.200,0.068,87.15,88.96,61.0,0.027,0.519,0.346,0.187,0.558,0.349,0.264,0.238,0,0,0,0,13,8,7,24,0,1,1.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.0,0.0,1.000,0.000,1,5,0,0,0.000,0,0,0.000,3,3,6,2,1,0,0,3,2,9.0,0.042,0.096,0.000,0.000,0.000,0.000,0.250,0.188,0.214,0.133,0.000,0.143,0.00,0.000,0.176,0.048,0.029
2,0029900011,1610612747,Los Angeles,Lakers,LAL,lakers,406,Shaquille,O'Neal,S. O'Neal,shaquille-oneal,C,NaN,NaN,34:26,120.3,96.8,26.0,23.5,0.050,0.50,3.6,0.147,0.258,0.200,7.1,0.450,0.463,0.355,89.58,88.52,64.0,0.118,0.559,0.305,0.141,0.462,0.283,0.186,0.269,3,4,0,16,8,14,9,24,2,0,1.000,0.000,0.783,0.087,0.000,0.000,0.217,0.130,0.696,0.556,0.444,0.0,0.0,0.556,0.444,9,20,0,0,0.000,5,11,0.455,5,8,13,1,1,2,2,

In [21]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,21995,1610612742,DAL,Dallas Mavericks,0029500011,1995-11-03,DAL @ SAN,W,240,103,42,89,0.472,4,12,0.333,15,23,0.652,15,34,49,16,7,7,17,30,NaN,1995-96
1,21995,1610612737,ATL,Atlanta Hawks,0029500002,1995-11-03,ATL vs. IND,L,240,106,35,73,0.479,6,14,0.429,30,47,0.638,14,13,27,21,12,4,12,32,NaN,1995-96
2,21995,1610612758,SAC,Sacramento Kings,0029500009,1995-11-03,SAC vs. MIN,W,240,95,33,67,0.493,7,13,0.538,22,33,0.667,6,31,37,21,12,6,21,26,NaN,1995-96
3,21995,1610612739,CLE,Cleveland Cavaliers,0029500010,1995-11-03,CLE @ ORL,L,240,88,30,63,0.476,9,16,0.563,19,33,0.576,6,25,31,20,9,4,10,21,NaN,1995-96
4,21995,1610612757,POR,Portland Trail Blazers,0029500007,1995-11-03,POR vs. VAN,L,240,80,29,81,0.358,5,22,0.227,17,30,0.567,23,28,51,20,11,3,26,25,NaN,1995-96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86077,42024,1610612754,IND,Indiana Pacers,0042400401,2025-06-05,IND @ OKC,W,240,111,39,82,0.476,18,39,0.462,15,21,0.714,13,43,56,24,1,7,24,22,1.0,2024-25
86078,42024,1610612760,OKC,Oklahoma City Thunder,0042400402,2025-06-08,OKC vs. IND,W,240,123,40,82,0.488,14,36,0.389,29,33,0.879,11,32,43,25,10,4,13,20,16.0,2024-25
86079,42024,1610612754,IND,Indiana Pacers,0042400402,2025-06-08,IND @ OKC,L,241,107,37,82,0.451,14,40,0.350,19,26,0.731,7,28,35,27,9,6,15,25,-16.0,2024-25
86080,42024,1610612760,OKC,Oklahoma City Thunder,0042400403,2025-06-11,OKC @ IND,L,239,107,37,79,0.468,10,22,0.455,23,30,0.767,9,33,42,16,6,4,17,20,-9.0,2024-25


# 🧼 Nettoyage des minutes jouées

In [22]:
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    try:
        parts = str(val).split(':')
        return int(parts[0]) + int(parts[1]) / 60 if len(parts) == 2 else float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['minutes'].apply(convert_minutes)

# Rename gameId and teamId in boxscores

In [23]:
rename_columns = {
    'gameId': 'GAME_ID',
    'teamId': 'TEAM_ID',
}

df_boxscores.rename(columns=rename_columns, inplace=True)


In [24]:
# cast 'GAME_ID', 'TEAM_ID' to string
df_games['GAME_ID'] = df_games['GAME_ID'].astype(str)
df_games['TEAM_ID'] = df_games['TEAM_ID'].astype(str)

df_boxscores['GAME_ID'] = df_boxscores['GAME_ID'].astype(str)
df_boxscores['TEAM_ID'] = df_boxscores['TEAM_ID'].astype(str)

In [25]:
print(df_games.dtypes)


SEASON_ID              int64
TEAM_ID               object
TEAM_ABBREVIATION     object
TEAM_NAME             object
GAME_ID               object
GAME_DATE             object
MATCHUP               object
WL                    object
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PLUS_MINUS           float64
SEASON                object
dtype: object


In [26]:
print(df_boxscores.dtypes)

GAME_ID                                object
TEAM_ID                                object
teamCity                               object
teamName                               object
teamTricode                            object
                                       ...   
percentageBlocksAllowed_usage         float64
percentagePersonalFouls_usage         float64
percentagePersonalFoulsDrawn_usage    float64
percentagePoints_usage                float64
MINUTES_PLAYED                        float64
Length: 101, dtype: object


# Merge GAME_DATE dans les boxscores et remove les duplicates créés

In [28]:
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])
df_boxscores = df_boxscores.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

df_boxscores = df_boxscores.drop_duplicates(subset=['GAME_ID', 'TEAM_ID', 'playerSlug'])

In [29]:
# display nat GAME_DATE in df_boxscores
# check for NaT values in GAME_DATE
nat_dates = df_boxscores[df_boxscores['GAME_DATE'].isna()]
nat_dates

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED,GAME_DATE


In [30]:
missing = df_boxscores[df_boxscores['GAME_DATE'].isna()][['GAME_ID', 'TEAM_ID']].drop_duplicates()

print(f"Exemples de lignes avec GAME_DATE NaT:")
print(missing.head(10))

# On regarde s’ils existent dans df_games
merged_check = missing.merge(df_games[['GAME_ID', 'TEAM_ID']], on=['GAME_ID', 'TEAM_ID'], how='left', indicator=True)
print(merged_check['_merge'].value_counts())


Exemples de lignes avec GAME_DATE NaT:
Empty DataFrame
Columns: [GAME_ID, TEAM_ID]
Index: []
_merge
left_only     0
right_only    0
both          0
Name: count, dtype: int64


In [31]:
print("Exemples d’ID manquants dans df_games")
print(missing[~missing.set_index(['GAME_ID', 'TEAM_ID']).index.isin(df_games.set_index(['GAME_ID', 'TEAM_ID']).index)])


Exemples d’ID manquants dans df_games
Empty DataFrame
Columns: [GAME_ID, TEAM_ID]
Index: []


# 🔄 Cast dynamique des colonnes numériques


In [32]:
numeric_cols = df_boxscores.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)

In [33]:
df_boxscores

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED,GAME_DATE
0,0029900011,1610612747,Los Angeles,Lakers,LAL,lakers,779,Glen,Rice,G. Rice,glen-rice,F,NaN,0.0,40:11,102.6,88.2,15.5,14.5,0.150,0.75,13.0,0.026,0.143,0.081,17.4,0.808,0.871,0.225,89.63,90.80,76.0,0.224,0.500,0.349,0.201,0.440,0.328,0.226,0.258,2,5,10,0,14,14,9,28,1,1,0.538,0.462,0.214,0.214,0.536,0.357,0.250,0.071,0.000,1.000,0.000,1.0,0.0,1.000,0.000,8,13,5,6,0.833,7,7,1.000,1,5,6,3,2,0,4,5,28,11.0,0.286,0.206,0.714,0.545,0.467,0.318,0.077,0.217,0.167,0.143,0.267,0.222,0.00,0.200,0.263,0.045,0.359,40.183333,1999-11-02
1,0029900011,1610612747,Los Angeles,Lakers,LAL,lakers,920,A.C.,Green,A. Green,ac-green,F,NaN,0.0,32:55,111.5,96.7,18.1,14.8,0.087,0.00,28.6,0.100,0.130,0.113,0.0,0.200,0.200,0.068,87.15,88.96,61.0,0.027,0.519,0.346,0.187,0.558,0.349,0.264,0.238,0,0,0,0,13,8,7,24,0,1,1.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.0,0.0,1.000,0.000,1,5,0,0,0.000,0,0,0.000,3,3,6,2,1,0,0,3,2,9.0,0.042,0.096,0.000,0.000,0.000,0.000,0.250,0.188,0.214,0.133,0.000,0.143,0.00,0.000,0.176,0.048,0.029,32.916667,1999-11-02
2,0029900011,1610612747,Los Angeles,Lakers,LAL,lakers,406,Shaquille,O'Neal,S. O'Neal,shaquille-oneal,C,NaN,0.0,34:26,120.3,96.8,26.0,23.5,0.050,0.50,3.6,0.147,0.258,0.200,7.1,0.450,0.463,0.355,89.58,88.52,64.0,0.118,0.559,0.305,0.141,0.462,0.283,0.186,0.269,3,4,0,16,8,14,9,24,2,0,1.000,0.000,0.783,0.087,0.000,0.000,0.217,0.130,0.696,0.55

# Build player features (players and top players absence for each game)

In [34]:
player_features_df = build_player_status_features(df_boxscores)

today = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')


os.makedirs(DATA_PLAYERS_DIR, exist_ok=True)
player_features_output = os.path.join(DATA_PLAYERS_DIR, f'player_features_{today}.csv')

# to csv
#player_features_df.to_csv(player_features_output, index=False)

In [35]:
player_features_df

,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
0,0029900011,1610612747,1999-11-02,779,1,0,0,0,0,0,0,37.7,-2.0,34.1,0.224,
1,0029900011,1610612747,1999-11-02,920,1,0,0,0,0,0,0,13.2,-1.5,5.4,0.027,
2,0029900011,1610612747,1999-11-02,406,1,0,0,0,0,0,0,41.1,0.5,27.1,0.118,
3,0029900011,1610612747,1999-11-02,965,1,0,0,0,0,0,0,25.1,-3.5,22.9,0.094,
4,0029900011,1610612747,1999-11-02,166,1,0,0,0,0,0,0,23.4,2.5,13.8,0.061,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3035625,0042400403,1610612754,2025-06-11,1628396,0,1,0,0,0,0,1,0.0,0.0,0.0,0.000,dnp - coach's decision
3035626,0042400403,1610612754,2025-06-11,1642277,0,1,0,0,0,0,1,0.0,0.0,0.0,0.000,dnp - coach's decision
3035627,0042400403,1610612754,2025-06-11,1630543,0,1,1,0,0,0,0,0.0,0.0,0.0,0.000,dnd - injury/illness
3035628,0042400403,1610612754,2025-06-11,201949,0,1,0,0,0,0,1,0.0,0.0,0.0,0.000,dnp - coach's decision


In [36]:
#print duplicates on 'GAME_ID' and 'TEAM_ID', personId
duplicates = player_features_df[player_features_df.duplicated(subset=['GAME_ID', 'TEAM_ID', 'personId'], keep=False)]
if not duplicates.empty:
    print("Duplicates found:")
    display(duplicates)

In [37]:
#player_features_df[player_features_df['is_absent'] == 1].sort_values(by='GAME_ID').head(50)

#same but filter with comment "DNP - Coach's Decision"

filtered = player_features_df[
    (player_features_df['is_absent'] == 1) &
    (~player_features_df['comment'].str.contains("coach's decision", na=False))
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(50))

filtered = player_features_df[
    (player_features_df['is_personal'] == 1) &
    (~player_features_df['comment'].str.contains("personal", na=False)) &
    (~player_features_df['comment'].str.contains("not with team", na=False)) 
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(50))




# display(player_features_df[
#     player_features_df['is_personal'] == 1 &
#     (~player_features_df['comment'].str.contains("nwt", na=False))
#     ].head(50))


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
28901,0049900028,1610612758,2000-05-02,895,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - sprained left ankle
28878,0049900030,1610612759,2000-05-02,1495,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,"dnd - lateral meniscus tear, left knee"
28879,0049900030,1610612759,2000-05-02,760,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - mid-right foot dislocation
28924,0049900032,1610612760,2000-05-03,1023,0,1,0,0,0,0,1,0.0,0.0,0.0,0.0,dnd - damaged facial nerve
28912,0049900032,1610612762,2000-05-03,228,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd- right hamstring
28949,0049900040,1610612760,2000-05-05,1023,0,1,0,0,0,0,1,0.0,0.0,0.0,0.0,dnd-did not dress - injured
28957,0049900040,1610612762,2000-05-05,228,0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd - did not dress - strained r. ham.
29021,0049900041,1610612748,2000-05-07,1934,0,1,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt -
29104,0049900042,1610612748,2000-05-09,1934,0,1,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt -
29206,0049900043,1610612748,2000-05-12,1934,0,1,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
243112,0020300765,1610612749,2004-02-17,1496,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family matters
243139,0020300777,1610612749,2004-02-18,1496,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family matters
244754,0020301147,1610612760,2004-04-10,2501,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - death in family
255836,0020400071,1610612765,2004-11-11,1112,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family matters
255965,0020400088,1610612765,2004-11-13,1112,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,"dnp - nwt, family matters"
262217,0020401200,1610612757,2005-04-17,1507,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family concerns
283337,0020500508,1610612760,2006-01-11,2501,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - excused absence attending funeral
346504,0020600508,1610612755,2007-01-09,2212,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - birth of child
397485,0020700039,1610612737,2007-11-04,1533,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family matters
414191,0020700731,1610612757,2008-02-08,200750,0,1,0,0,0,1,0,0.0,0.0,0.0,0.0,nwt - family matter


In [38]:

# Analyse des statuts des joueurs
total_rows = len(player_features_df)

# Comptage des cas
n_present = player_features_df['is_present'].sum()
n_absent = player_features_df['is_absent'].sum()
n_injured = player_features_df['is_injured'].sum()

# is_resting
# is_suspended
# is_personal

n_resting = player_features_df['is_resting'].sum()
n_suspended = player_features_df['is_suspended'].sum()
n_personal = player_features_df['is_personal'].sum()

# Lignes incohérentes : aucune des colonnes n'est True

mask_no_status = (
    (player_features_df['is_present'] == 0) &
    (player_features_df['is_absent'] == 0) &
    (player_features_df['is_injured'] == 0) &
    (player_features_df['is_resting'] == 0) &
    (player_features_df['is_suspended'] == 0) &
    (player_features_df['is_personal'] == 0)
)


n_inconsistent = mask_no_status.sum()

# Lignes avec plusieurs statuts à la fois (logiquement impossible)
mask_multiple_status = (
    player_features_df[['is_present', 'is_absent', 'is_injured']].sum(axis=1) > 1
)
n_multiple = mask_multiple_status.sum()

print(f"✅ Analyse des statuts des joueurs sur {total_rows} lignes")
print(f" - Joueurs présents : {n_present}")
print(f" - Joueurs absents : {n_absent}")
print(f" - Joueurs blessés : {n_injured}")
print(f" - Joueurs en repos : {n_resting}")
print(f" - Joueurs suspendus : {n_suspended}")
print(f" - Joueurs pour raisons personnelles : {n_personal}")
print(f"❌ Lignes sans statut défini : {n_inconsistent}")
print(f"⚠️ Lignes avec plusieurs statuts actifs : {n_multiple}")

display(player_features_df[mask_no_status].head(10))



✅ Analyse des statuts des joueurs sur 782356 lignes
 - Joueurs présents : 642800
 - Joueurs absents : 139556
 - Joueurs blessés : 22195
 - Joueurs en repos : 393
 - Joueurs suspendus : 1063
 - Joueurs pour raisons personnelles : 1056
❌ Lignes sans statut défini : 0
⚠️ Lignes avec plusieurs statuts actifs : 22195


,GAME_ID,TEAM_ID,GAME_DATE,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,is_absent_other,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment


# ⚙️ Aggrégation des data Player par équipe et match


In [42]:
# df_team_players = aggregate_team_player_features(player_features_df, top_n=10)

# display(df_team_players.head(10))
# display(df_team_players.tail(10))

# #display some lines with absent players
# absent_players = df_team_players[df_team_players['num_injured'] >= 1]
# absent_players 

# Identification des top joueurs sur l'ensemble du dataset
df_top_players = identify_historical_top_players(player_features_df)
print(f"✅ {df_top_players['is_historical_top'].sum()} top joueurs identifiés sur {len(df_top_players)} joueurs.")


df_agg_actual = aggregate_actual_team_features(player_features_df)
print(f"✅ {df_agg_actual.shape[0]} lignes générées dans l'aggregation actuelle par équipe.")

df_agg_top_abs = flag_top_players_absences(player_features_df, df_top_players)
print(f"✅ {df_agg_top_abs['top_player_absent'].sum()} absences de top joueurs détectées.")

df_team_features_final = df_agg_actual.merge(df_agg_top_abs, on=["GAME_ID", "TEAM_ID"], how="left")
df_team_features_final.fillna(0, inplace=True)

#add flags has_top_absent flag to use it as input and not roll it to see in analyse how much it helps to scrap this data before match
df_team_features_final['has_top_absent'] = df_team_features_final['top_player_absent'].apply(lambda x: 1 if x > 0 else 0)
df_team_features_final['has_absent'] = df_team_features_final['num_absent'].apply(lambda x: 1 if x > 0 else 0)


#sort by GAME_ID, TEAM_ID and GAME_DATE
df_team_features_final.sort_values(by=['GAME_ID', 'TEAM_ID', 'GAME_DATE'], inplace=True)


print(f"✅ Fusion réussie. Shape finale : {df_team_features_final.shape}")


✅ 319691 top joueurs identifiés sur 319691 joueurs.
✅ 66201 lignes générées dans l'aggregation actuelle par équipe.
✅ 19709 absences de top joueurs détectées.
✅ Fusion réussie. Shape finale : (66201, 27)


In [44]:
# display(df_team_features_final.head(10))
# display(df_team_features_final.tail(10))

# top_player_absence_rate
# top_player_injury_rate
# top_player_resting_rate
# top_player_suspension_rate
# top_player_personal_rate

# display(df_team_features_final[df_team_features_final['top_player_absent_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_injury_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_resting_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_suspension_rate'] > 0])
# display(df_team_features_final[df_team_features_final['top_player_personal_rate'] > 0])
display(df_team_features_final)


,GAME_ID,TEAM_ID,GAME_DATE,player_perf_score_mean,player_perf_score_sum,num_present,num_absent,num_injured,num_suspended,num_resting,num_personal,num_absent_other,top_player_count,top_player_absent,top_player_injured,top_player_resting,top_player_suspended,top_player_personal,top_player_absent_other,top_player_absent_rate,top_player_injury_rate,top_player_resting_rate,top_player_suspension_rate,top_player_personal_rate,top_player_absent_other_rate,has_top_absent,has_absent
0,0020000001,1610612752,2000-10-31,10.450000,125.4,12,0,0,0,0,0,0,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
1,0020000001,1610612755,2000-10-31,15.658333,187.9,12,0,0,0,0,0,0,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2,0020000002,1610612739,2000-10-31,13.866667,166.4,11,1,0,0,0,0,1,5,1,0,0,0,0,1,0.2,0.0,0.0,0.0,0.0,0.2,1,1
3,0020000002,1610612751,2000-10-31,14.950000,179.4,10,2,0,0,0,0,2,5,1,0,0,0,0,1,0.2,0.0,0.0,0.0,0.0,0.2,1,1
4,0020000003,1610612753,2000-10-31,14.616667,175.4,10,2,0,0,0,0,2,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66196,0052400131,1610612758,2025-04-16,14.253846,185.3,12,1,0,0,0,0,1,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
66197,0052400201,1610612737,2025-04-18,18.150000,217.8,9,3,1,0,0,0,2,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
66198,0052400201,1610612748,2025-04-18,15.686667,235.3,9,6,0,0,0,1,5,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1
66199,0052400211,1610612742,2025-04-18,14.438462,187.7,11,2,0,0,0,0,2,5,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


# ⚙️ Aggrégation des data Boxscore purs par équipe et match


## 📊 Agrégation des données par équipe et match


In [49]:

# 1. Agrégation par somme
group_keys = ['GAME_ID', 'TEAM_ID']
sum_agg = df_boxscores[group_keys + cols_to_sum].copy()
sum_agg = sum_agg.groupby(group_keys).sum().reset_index()

# 2. Agrégation pondérée par les minutes jouées
weighted_agg = compute_weighted_mean_features(df_boxscores, group_keys, cols_to_weighted_avg, weight_col='MINUTES_PLAYED')

# 3. Fusion des deux agrégats
team_match_stats = pd.merge(sum_agg, weighted_agg, on=group_keys, how='left')

# 4. Identifier l'équipe adverse
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
def get_opponent(row):
    teams = teams_in_game.get(row['GAME_ID'], [])
    opps = [tid for tid in teams if tid != row['TEAM_ID']]
    return opps[0] if opps else None

team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(get_opponent, axis=1)


In [ ]:
count = 0

for game_id, teams in teams_in_game.items():
    if len(teams) < 2:
        #print(f"Attention : GAME_ID {game_id} a moins de 2 équipes : {teams}")
        count += 1
        
print(f"Nombre de GAME_ID avec moins de 2 équipes : {count}")



Attention : GAME_ID 0020300003 a moins de 2 équipes : ['1610612747']
Attention : GAME_ID 0020300017 a moins de 2 équipes : ['1610612745']
Attention : GAME_ID 0020300021 a moins de 2 équipes : ['1610612749']
Attention : GAME_ID 0020300032 a moins de 2 équipes : ['1610612749']
Attention : GAME_ID 0020300035 a moins de 2 équipes : ['1610612759']
Attention : GAME_ID 0020300053 a moins de 2 équipes : ['1610612764']
Attention : GAME_ID 0020300069 a moins de 2 équipes : ['1610612753']
Attention : GAME_ID 0020300089 a moins de 2 équipes : ['1610612760']
Attention : GAME_ID 0020300118 a moins de 2 équipes : ['1610612760']
Attention : GAME_ID 0020300122 a moins de 2 équipes : ['1610612761']
Attention : GAME_ID 0020300138 a moins de 2 équipes : ['1610612747']
Attention : GAME_ID 0020300144 a moins de 2 équipes : ['1610612739']
Attention : GAME_ID 0020300291 a moins de 2 équipes : ['1610612760']
Attention : GAME_ID 0020300292 a moins de 2 équipes : ['1610612748']
Attention : GAME_ID 0020300303 a m

# 🔁 Ajout des colonnes OPP_ avec les data de l'adversaire

In [51]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]
df_opp = team_match_stats.rename(columns={col: f"OPP_{col}" for col in team_cols}).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'})
match_dataset = pd.merge(
    team_match_stats,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [52]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'fieldGoalsMade_traditional',
       'fieldGoalsAttempted_traditional', 'threePointersMade_traditional',
       'threePointersAttempted_traditional', 'freeThrowsMade_traditional',
       'freeThrowsAttempted_traditional', 'reboundsOffensive_traditional',
       'reboundsDefensive_traditional',
       ...
       'OPP_percentageReboundsTotal_usage', 'OPP_percentageAssists_usage',
       'OPP_percentageTurnovers_usage', 'OPP_percentageSteals_usage',
       'OPP_percentageBlocks_usage', 'OPP_percentageBlocksAllowed_usage',
       'OPP_percentagePersonalFouls_usage',
       'OPP_percentagePersonalFoulsDrawn_usage', 'OPP_percentagePoints_usage',
       'OPP_plusMinusPoints_traditional'],
      dtype='object', length=165)

# 🏠 Ajout IS_HOME et IS_WIN


In [53]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,21995,1610612742,DAL,Dallas Mavericks,0029500011,1995-11-03,DAL @ SAN,W,240,103,42,89,0.472,4,12,0.333,15,23,0.652,15,34,49,16,7,7,17,30,NaN,1995-96
1,21995,1610612737,ATL,Atlanta Hawks,0029500002,1995-11-03,ATL vs. IND,L,240,106,35,73,0.479,6,14,0.429,30,47,0.638,14,13,27,21,12,4,12,32,NaN,1995-96
2,21995,1610612758,SAC,Sacramento Kings,0029500009,1995-11-03,SAC vs. MIN,W,240,95,33,67,0.493,7,13,0.538,22,33,0.667,6,31,37,21,12,6,21,26,NaN,1995-96
3,21995,1610612739,CLE,Cleveland Cavaliers,0029500010,1995-11-03,CLE @ ORL,L,240,88,30,63,0.476,9,16,0.563,19,33,0.576,6,25,31,20,9,4,10,21,NaN,1995-96
4,21995,1610612757,POR,Portland Trail Blazers,0029500007,1995-11-03,POR vs. VAN,L,240,80,29,81,0.358,5,22,0.227,17,30,0.567,23,28,51,20,11,3,26,25,NaN,1995-96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86077,42024,1610612754,IND,Indiana Pacers,0042400401,2025-06-05,IND @ OKC,W,240,111,39,82,0.476,18,39,0.462,15,21,0.714,13,43,56,24,1,7,24,22,1.0,2024-25
86078,42024,1610612760,OKC,Oklahoma City Thunder,0042400402,2025-06-08,OKC vs. IND,W,240,123,40,82,0.488,14,36,0.389,29,33,0.879,11,32,43,25,10,4,13,20,16.0,2024-25
86079,42024,1610612754,IND,Indiana Pacers,0042400402,2025-06-08,IND @ OKC,L,241,107,37,82,0.451,14,40,0.350,19,26,0.731,7,28,35,27,9,6,15,25,-16.0,2024-25
86080,42024,1610612760,OKC,Oklahoma City Thunder,0042400403,2025-06-11,OKC @ IND,L,239,107,37,79,0.468,10,22,0.455,23,30,0.767,9,33,42,16,6,4,17,20,-9.0,2024-25


In [54]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [55]:
# Assurer le format datetime pour GAME_DATE
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])



# Merge avec les infos de match (MATCHUP, SEASON)
match_dataset = match_dataset.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP', 'SEASON','GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

In [56]:

print("match_dataset after merge")
display(match_dataset)

match_dataset after merge


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [57]:

#display lines with nan values on matchup
print("Lines with NaN in MATCHUP:")
display(match_dataset[match_dataset['MATCHUP'].isna()])

Lines with NaN in MATCHUP:


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

In [58]:



# Définir si l'équipe joue à domicile
match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs').astype(int)

# Calcul du résultat (win) et écart de points
match_dataset['IS_WIN'] = (match_dataset['points_traditional'] > match_dataset['OPP_points_traditional']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['points_traditional'] - match_dataset['OPP_points_traditional']

# Conversion GAME_DATE et tri
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)


In [59]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

# SPECIAL FOR ODDS : Keep only season past 2010-11


In [ ]:
# # get first line with season 2010-11 and cut all lines before
# first_season = match_dataset[match_dataset['SEASON'] == '2010-11'].index[0]
# match_dataset = match_dataset.iloc[first_season:].reset_index(drop=True)
# match_dataset

# Merge Odds with dataset

In [64]:

def match_odds_with_dataset_test(odds_df, nba_df):
    
    team_id_map = get_team_mapping_id()
    
    nba_df = nba_df.copy()
    odds_df = odds_df.copy()

    # Conversion explicite des ID en str
    nba_df["TEAM_ID"] = nba_df["TEAM_ID"].astype(str)
    nba_df["OPP_TEAM_ID"] = nba_df["OPP_TEAM_ID"].astype(str)
    team_id_map = {str(k): v for k, v in team_id_map.items()}

    nba_df["TEAM_NAME"] = nba_df["TEAM_ID"].map(team_id_map).apply(clean_team_name)
    nba_df["OPPONENT_NAME"] = nba_df["OPP_TEAM_ID"].map(team_id_map).apply(clean_team_name)
    nba_df["GAME_DATE"] = pd.to_datetime(nba_df["GAME_DATE"]).dt.date

    odds_df["home_team"] = odds_df["home_team"].apply(clean_team_name)
    odds_df["away_team"] = odds_df["away_team"].apply(clean_team_name)
    odds_df["date"] = pd.to_datetime(odds_df["date"]).dt.date

    nba_df["match_key"] = nba_df.apply(
        lambda row: (row["GAME_DATE"], row["TEAM_NAME"], row["OPPONENT_NAME"])
        if row["IS_HOME"] == 1
        else (row["GAME_DATE"], row["OPPONENT_NAME"], row["TEAM_NAME"]),
        axis=1,
    )

    keys_full = []
    for shift in [-1, 0, 1]:
        shifted = odds_df.copy()
        shifted["match_key"] = shifted.apply(
            lambda row: (row["date"] + timedelta(days=shift), row["home_team"], row["away_team"]),
            axis=1
        )
        keys_full.append(shifted)

    odds_full = pd.concat(keys_full, ignore_index=True)
    odds_full = odds_full.drop_duplicates(subset=["match_key"])

    # print("\nExemples de clés de match dans odds_df (tolérance date):")
    # print(odds_full["match_key"].drop_duplicates().head())
    # print("\nExemples de clés de match dans nba_df:")
    # print(nba_df["match_key"].drop_duplicates().head())


    merged = pd.merge(odds_full, nba_df, on="match_key", how="right")
    
    # Attribution claire des cotes à chaque ligne équipe
    merged["ODDS"] = merged.apply(
        lambda row: row["home_odds"] if row["IS_HOME"] == 1 else row["away_odds"], axis=1
    )
    merged["OPP_ODDS"] = merged.apply(
        lambda row: row["away_odds"] if row["IS_HOME"] == 1 else row["home_odds"], axis=1
    )
    
    # print(f"\nNombre de lignes fusionnées: {len(merged)}")
    # print(f"Nombre de correspondances réussies: {merged['TEAM_ID'].notna().sum()}")
    # print(f"Nombre de correspondances échouées: {merged['TEAM_ID'].isna().sum()}")

    # print("\nIDs manquants TEAM_ID:", nba_df[~nba_df["TEAM_ID"].isin(team_id_map.keys())]["TEAM_ID"].unique())
    # print("IDs manquants OPP_TEAM_ID:", nba_df[~nba_df["OPP_TEAM_ID"].isin(team_id_map.keys())]["OPP_TEAM_ID"].unique())

    # Nettoyage des données fusionnées
    merged = clean_merged_matches(merged)

    return merged


In [65]:
all_odds_df = merge_odds_csv_files(DATA_ODDS_HISTORY_DIR)

match_dataset = match_odds_with_dataset_test(all_odds_df, match_dataset)



In [ ]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentagePointsFast

: 

# Merge match_dataset with df_team_features_final on TEAM and GAME

In [ ]:

#team id as str
match_dataset['TEAM_ID'] = match_dataset['TEAM_ID'].astype(str)
match_dataset['GAME_ID'] = match_dataset['GAME_ID'].astype(str)

#date as datetime
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])

df_team_features_final['TEAM_ID'] = df_team_features_final['TEAM_ID'].astype(str)
df_team_features_final['GAME_ID'] = df_team_features_final['GAME_ID'].astype(str)
df_team_features_final['GAME_DATE'] = pd.to_datetime(df_team_features_final['GAME_DATE'])



# Merge match_dataset with df_team_features_final on TEAM and GAME, keep GAME_DATE
match_dataset = match_dataset.merge(
    df_team_features_final[['GAME_ID', 'TEAM_ID', 'GAME_DATE'] + df_team_features_final.columns.difference(['GAME_ID', 'TEAM_ID', 'GAME_DATE']).tolist()],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)



# match_dataset_ = match_dataset.merge(
#     df_team_features_final,
#     on=['GAME_ID', 'TEAM_ID'],
#     how='left'
# )
#remove duplicates
match_dataset = match_dataset.drop_duplicates(subset=['GAME_ID', 'TEAM_ID'])



In [ ]:
#sort by GAME_DATE and GAME_ID
#match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)



#print GAME_DATE_x and GAME_DATE_y to check if they are the same
print("GAME_DATE_x and GAME_DATE_y are the same:", (match_dataset['GAME_DATE_x'] == match_dataset['GAME_DATE_y']).all())

#drop GAME_DATE_x and GAME_DATE_y to GAME_DATE
match_dataset['GAME_DATE'] = match_dataset['GAME_DATE_x']
match_dataset = match_dataset.drop(columns=['GAME_DATE_x', 'GAME_DATE_y'])



#display cols in common match_dataset and df_team_features_final
common_cols = set(match_dataset.columns).intersection(set(df_team_features_final.columns))
print("Common columns between match_dataset and df_team_features_final:")
display(common_cols)


GAME_DATE_x and GAME_DATE_y are the same: True
Common columns between match_dataset and df_team_features_final:


{'GAME_DATE',
 'GAME_ID',
 'TEAM_ID',
 'has_absent',
 'has_top_absent',
 'num_absent',
 'num_absent_other',
 'num_injured',
 'num_personal',
 'num_present',
 'num_resting',
 'num_suspended',
 'player_perf_score_mean',
 'player_perf_score_sum',
 'top_player_absent',
 'top_player_absent_other',
 'top_player_absent_other_rate',
 'top_player_absent_rate',
 'top_player_count',
 'top_player_injured',
 'top_player_injury_rate',
 'top_player_personal',
 'top_player_personal_rate',
 'top_player_resting',
 'top_player_resting_rate',
 'top_player_suspended',
 'top_player_suspension_rate'}

# Reorder columns for visualisation

In [ ]:
cols_first = ['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN','POINT_DIFF','has_absent','has_top_absent','num_absent','top_player_absent', 'points_traditional','ODDS','OPP_ODDS']
other_cols = [col for col in match_dataset.columns if col not in cols_first]
match_dataset = match_dataset[cols_first + other_cols]
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,has_absent,has_top_absent,num_absent,top_player_absent,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt

# Add OPP cols for player stats

In [ ]:
opp_cols = [f"OPP_{col}" for col in cols_player_stats]
df_opp = match_dataset.rename(columns={col: f"OPP_{col}" for col in cols_player_stats})

match_dataset = pd.merge(
    match_dataset,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [ ]:
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,has_absent,has_top_absent,num_absent,top_player_absent,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt

# 🚀 Features avancées


In [ ]:

# Ajouter les features avancées aux stats de match
#match_dataset = add_advanced_boxscore_features(match_dataset)

# MOVED TO CONFIG
# features_to_roll = cols_to_sum + cols_to_weighted_avg
# features_to_roll += [f"OPP_{col}" for col in cols_to_sum + cols_to_weighted_avg]

# Calcul des features glissantes shiftées
match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], features_to_roll, N_LIST, method="ewm")

match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], top_player_features_to_roll, N_LIST_TOP, method="ewm", apply_log=True)


match_dataset = compute_winrates(match_dataset, "TEAM_ID", "IS_WIN", "IS_HOME", N_LIST)
match_dataset = compute_win_ratio(match_dataset, "TEAM_ID", "IS_WIN", N_LIST)

match_dataset["IS_WIN_SHIFTED"] = match_dataset.groupby("TEAM_ID")["IS_WIN"].shift(1).fillna(0).astype(int)
match_dataset["WIN_STREAK"] = match_dataset.groupby("TEAM_ID").apply(
    lambda x: compute_win_streak(x, "TEAM_ID", "IS_WIN_SHIFTED")).reset_index(level=0, drop=True)
match_dataset = compute_side_win_streak(match_dataset, win_shifted_col="IS_WIN_SHIFTED")


# Calcul des jours de repos pour l'équipe et l'adversaire
match_dataset["DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "TEAM_ID")
match_dataset["OPP_DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "OPP_TEAM_ID")

# Avantage de repos
match_dataset["REST_ADVANTAGE"] = (
    match_dataset["DAYS_SINCE_LAST_GAME"] - match_dataset["OPP_DAYS_SINCE_LAST_GAME"]
)


match_dataset = compute_rolling_rest_advantage(
    match_dataset, "TEAM_ID", "IS_HOME", "REST_ADVANTAGE", N_LIST
)

match_dataset = compute_home_away_pts(
    match_dataset, "TEAM_ID", "IS_HOME", "points_traditional", "OPP_points_traditional", N_LIST
)

# match_dataset = rename_pts_against_columns(match_dataset)

match_dataset = compute_h2h(match_dataset, N_LIST)
# match_dataset = compute_h2h_pts_margin(match_dataset, N_LIST)
match_dataset = compute_h2h_season(match_dataset)
match_dataset = compute_h2h_streak(match_dataset)

match_dataset = compute_elo(match_dataset)
match_dataset = compute_elo_season(match_dataset)

match_dataset = convert_elos_to_elo_diff(match_dataset)


e:\Documents_\Dev\NBA_Predictor\src\feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[roll_col] = (
e:\Documents_\Dev\NBA_Predictor\src\feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[roll_col] = (
e:\Documents_\Dev\NBA_Predictor\src\feature_builder.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fr

In [ ]:
pd.options.display.max_columns = None
display(match_dataset)

GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0021000001 2010-10-26  1610612738  1610612748      1.0     1.0   
1      0021000001 2010-10-26  1610612748  1610612738      0.0     0.0   
2      0021000002 2010-10-26  1610612756  1610612757      0.0     0.0   
3      0021000002 2010-10-26  1610612757  1610612756      1.0     1.0   
4      0021000003 2010-10-26  1610612745  1610612747      0.0     0.0   
...           ...        ...         ...         ...      ...     ...   
38076  0042400315 2025-05-28  1610612760  1610612750      1.0     1.0   
38077  0042400305 2025-05-29  1610612752  1610612754      1.0     1.0   
38078  0042400305 2025-05-29  1610612754  1610612752      0.0     0.0   
38079  0042400306 2025-05-31  1610612752  1610612754      0.0     0.0   
38080  0042400306 2025-05-31  1610612754  1610612752      1.0     1.0   

       POINT_DIFF  has_absent  has_top_absent  num_absent  top_player_absent  \
0             8.0           1               0           3                  0   
1            -8.0           1               1           3                  2   
2           -14.0           1               0           2                  0   
3            14.0           1               0           2                  0   
4            -2.0           1               1           2                  1   
...           ...         ...             ...         ...                ...   
38076        30.0           0               0           0                  0   
38077        17.0           1               0           4                  0   
38078       -17.0           1               0           2                  0   
38079       -17.0           1               0           2                  0   
38080        17.0           1               0           2                  0   

       points_traditional  ODDS  OPP_ODDS  fieldGoalsMade_traditional  \
0                    88.0  1.88      1.70                        32.0   
1                    80.0  1.70      1.88                        27.0   
2                    92.0  3.30      1.22                        36.0   
3                   106.0  1.22      3.30                        43.0   
4                   110.0  3.50      1.20                        38.0   
...                   ...   ...       ...                         ...   
38076               124.0  1.25      3.91                        46.0   
38077               111.0  1.54      2.46                        44.0   
38078                94.0  2.46      1.54                        30.0   
38079               108.0  2.38      1.58                        41.0   
38080               125.0  1.58      2.38                        46.0   

       fieldGoalsAttempted_traditional  threePointersMade_traditional  \
0                                 69.0                            8.0   
1                                 74.0                            8.0   
2                                 74.0                            9.0   
3                                 93.0                           10.0   
4                                 91.0                            8.0   
...                                ...                            ...   
38076                             88.0                           14.0   
38077                             89.0                            8.0   
38078                             74.0                           10.0   
38079                             86.0                            9.0   
38080                             85.0                           17.0   

       threePointersAttempted_traditional  freeThrowsMade_traditional  \
0                                    16.0                        16.0   
1                                    20.0                        18.0   
2                                    19.0                        11.0   
3                                    20.0                        10.0   
4                                    20.0            

# 🧽 Nettoyage et sauvegarde


In [ ]:


final_date = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
raw_save_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')
clean_save_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

os.makedirs(DATA_FINAL_DATASET_DIR, exist_ok=True)
os.makedirs(DATA_FINAL_CLEANED_DATASET_DIR, exist_ok=True)

match_dataset.to_csv(raw_save_path, index=False)

cols_to_drop = features_to_roll+top_player_features_to_roll+COLS_MATCH_REAL
#remove num_absent and top_player_absent from cols_to_drop to see if we can use them has input
cols_to_drop = [col for col in cols_to_drop if col not in ['num_absent', 'top_player_absent']]


final_cleaned = match_dataset.drop(columns=cols_to_drop, errors='ignore')
final_cleaned.to_csv(clean_save_path, index=False)

print(f"✅ Fichier brut : {raw_save_path}")
print(f"✅ Fichier clean : {clean_save_path}")

final_cleaned

✅ Fichier brut : data\final_dataset\nba_features_final_2025-06-12_20-02-49.csv
✅ Fichier clean : data\final_cleaned_dataset\nba_features_cleaned_final_2025-06-12_20-02-49.csv


GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0021000001 2010-10-26  1610612738  1610612748      1.0     1.0   
1      0021000001 2010-10-26  1610612748  1610612738      0.0     0.0   
2      0021000002 2010-10-26  1610612756  1610612757      0.0     0.0   
3      0021000002 2010-10-26  1610612757  1610612756      1.0     1.0   
4      0021000003 2010-10-26  1610612745  1610612747      0.0     0.0   
...           ...        ...         ...         ...      ...     ...   
38076  0042400315 2025-05-28  1610612760  1610612750      1.0     1.0   
38077  0042400305 2025-05-29  1610612752  1610612754      1.0     1.0   
38078  0042400305 2025-05-29  1610612754  1610612752      0.0     0.0   
38079  0042400306 2025-05-31  1610612752  1610612754      0.0     0.0   
38080  0042400306 2025-05-31  1610612754  1610612752      1.0     1.0   

       POINT_DIFF  has_absent  has_top_absent  num_absent  top_player_absent  \
0             8.0           1               0           3                  0   
1            -8.0           1               1           3                  2   
2           -14.0           1               0           2                  0   
3            14.0           1               0           2                  0   
4            -2.0           1               1           2                  1   
...           ...         ...             ...         ...                ...   
38076        30.0           0               0           0                  0   
38077        17.0           1               0           4                  0   
38078       -17.0           1               0           2                  0   
38079       -17.0           1               0           2                  0   
38080        17.0           1               0           2                  0   

       ODDS  OPP_ODDS   SEASON  ROLL_fieldGoalsMade_traditional_3  \
0      1.88      1.70  2010-11                                NaN   
1      1.70      1.88  2010-11                                NaN   
2      3.30      1.22  2010-11                                NaN   
3      1.22      3.30  2010-11                                NaN   
4      3.50      1.20  2010-11                                NaN   
...     ...       ...      ...                                ...   
38076  1.25      3.91  2024-25                          43.687438   
38077  1.54      2.46  2024-25                          37.928151   
38078  2.46      1.54  2024-25                          42.227196   
38079  2.38      1.58  2024-25                          40.964076   
38080  1.58      2.38  2024-25                          36.113598   

       ROLL_fieldGoalsMade_traditional_5  ROLL_fieldGoalsMade_traditional_10  \
0                                    NaN                                 NaN   
1                                    NaN                                 NaN   
2                                    NaN                                 NaN   
3                                    NaN                                 NaN   
4                                    NaN                                 NaN   
...                                  ...                                 ...   
38076                          42.975442                           42.674003   
38077                          38.496268                           38.913265   
38078                          42.415353                           42.844523   
38079                          40.330846                           39.838126   
38080                          38.276902                           40.509156   

       ROLL_fieldGoalsMade_traditional_25  ROLL_fieldGoalsMade_traditional_50  \
0                                     NaN                                 NaN   
1                                     NaN                                 NaN   
2                                     NaN                                 NaN   
3                                     NaN                 

In [ ]:
final_cleaned.columns

Index(['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN',
       'POINT_DIFF', 'has_absent', 'has_top_absent', 'num_absent',
       ...
       'H2H_LAST_100_COUNT', 'H2H_LAST_200_DIFF', 'H2H_LAST_200_WINRATE',
       'H2H_LAST_200_COUNT', 'H2H_SEASON_WINS', 'H2H_SEASON_MATCHES',
       'H2H_SEASON_WINRATE', 'H2H_WIN_STREAK', 'ELO_DIFF', 'ELO_DIFF_SEASON'],
      dtype='object', length=1464)

In [ ]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-06-12 20:05:27.066582
Total time:  0:13:49.732725
